# Healthcare Fraud Detection - Exploratory Data Analysis

## Purpose

This notebook explores patterns, distributions, relationships, and visualizations in the healthcare fraud detection dataset.

*Author: Brandon Ismalej*

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
from pathlib import Path

In [38]:
warnings.filterwarnings("ignore")

# Display and plot defaults
#pd.set_option("display.max_columns", None)
#pd.set_option("display.float_format", "{:,.2f}".format)
#sns.set_theme(style="whitegrid", context="notebook")
%matplotlib inline

In [ ]:
# Path to raw data
RAW_DATA_PATH = "../data/raw/healthcare_fraud_detection.csv"

In [29]:
# Import data
df = pd.read_csv(RAW_DATA_PATH)

df.head()


,Provider_ID,Claim_ID,Patient_Age,Patient_Gender,Diagnosis_Code,Procedure_Code,Claim_Amount,Approved_Amount,Insurance_Type,Claim_Submission_Date,Days_Between_Service_and_Claim,Number_of_Claims_Per_Provider_Monthly,Provider_Specialty,Patient_State,Claim_Status,Is_Fraud,Length_of_Stay,Visit_Type,Chronic_Condition_Flag,Prior_Visits_12m
0,P0052,C0000000,37,Male,I25.10,36415,443.51,393.16,Medicaid,2024-09-01,13,70,Cardiology,NY,Approved,0,0,Outpatient,1,2.00
1,P0121,C0000001,21,Female,E11.9,99213,467.50,461.33,Self-Pay,2022-09-05,5,62,General Practice,IL,Pending,0,5,Inpatient,1,2.00
2,P0140,C0000002,78,Female,J06.9,93000,591.69,530.06,Medicaid,2022-04-11,29,60,Cardiology,IL,Pending,0,5,Inpatient,1,3.00
3,P0202,C0000003,65,Male,I10,93000,235.15,189.11,Private,2023-10-11,22,70,General Practice,TX,Approved,0,0,Emergency,0,5.00
4,P0135,C0000004,36,Male,M54.5,85025,487.96,369.91,Private,2023-09-05,21,67,Pulmonology,PA,Approved,0,5,Inpatient,0,4.00


In [ ]:
# Summary stats for numeric columns
df.describe()

In [17]:
# Check missingness
print(df.isna().sum())

# Show percentages of missingness per column
missing = df.isna().mean().mul(100).round(2)
missing[missing > 0].sort_values(ascending=False)

Provider_ID                                0
Claim_ID                                   0
Patient_Age                                0
Patient_Gender                             0
Diagnosis_Code                             0
Procedure_Code                             0
Claim_Amount                               0
Approved_Amount                            0
Insurance_Type                           350
Claim_Submission_Date                      0
Days_Between_Service_and_Claim             0
Number_of_Claims_Per_Provider_Monthly      0
Provider_Specialty                       350
Patient_State                              0
Claim_Status                               0
Is_Fraud                                   0
Length_of_Stay                             0
Visit_Type                                 0
Chronic_Condition_Flag                     0
Prior_Visits_12m                         350
dtype: int64


Insurance_Type       3.50
Provider_Specialty   3.50
Prior_Visits_12m     3.50
dtype: float64

In [25]:
# For columns with missing values, view what other values look like
cols_with_na = df.columns[df.isna().any()]

for col in cols_with_na:
    print(df[col].value_counts(dropna=False).head(10))
    print()

Insurance_Type
Medicare    2425
Medicaid    2419
Private     2405
Self-Pay    2401
NaN          350
Name: count, dtype: int64

Provider_Specialty
Internal Medicine    2019
General Practice     1878
Orthopedics          1616
Neurology            1516
Cardiology           1366
Pulmonology          1255
NaN                   350
Name: count, dtype: int64

Prior_Visits_12m
3.00    2177
2.00    2076
4.00    1696
1.00    1436
5.00     999
6.00     487
0.00     460
NaN      350
7.00     214
8.00      74
Name: count, dtype: int64



In [ ]:
# Plot missing values by fraud label
cols = ["Insurance_Type", "Provider_Specialty", "Prior_Visits_12m"]

missing_by_fraud = df[cols].isna().groupby(df["Is_Fraud"]).sum().T
missing_by_fraud.columns = ["No Fraud", "Fraud"]

missing_by_fraud.plot(kind="bar")
plt.ylabel("Missing count")
plt.title("Missing values by fraud label")
plt.xticks(rotation=0)
plt.show()

In [ ]:
# Plot distribution of numeric columns by fraud label
num_cols = df.select_dtypes("number").columns.drop("Is_Fraud")

fig, axes = plt.subplots(3, 3, figsize=(15, 12))

for ax, col in zip(axes.flat, num_cols):
    sns.histplot(data=df, x=col, hue="Is_Fraud", stat="density", common_norm=False, ax=ax)
    ax.set_title(col)

plt.tight_layout()
plt.show()


In [ ]:
# Plot sample count of fraud vs no fraud
counts = df["Is_Fraud"].value_counts()

counts.plot(
    kind="pie",
    labels=["No Fraud", "Fraud"],
    autopct=lambda p: f"{p:.1f}%\n({p * counts.sum() / 100:.0f})",
    figsize=(4, 4),
)
plt.ylabel("")
plt.title("Sample Count: Fraud vs No Fraud")
plt.show()

In [ ]:
# Plot counts of categorical columns by fraud label
cat_cols = ["Patient_Gender", "Diagnosis_Code", "Insurance_Type", "Provider_Specialty",
            "Patient_State", "Claim_Status", "Visit_Type"]

fig, axes = plt.subplots(3, 3, figsize=(15, 12))

for ax, col in zip(axes.flat, cat_cols):
    sns.countplot(data=df, x=col, hue="Is_Fraud", ax=ax)
    ax.set_title(col)
    ax.tick_params(axis="x", rotation=45)

for ax in axes.flat[len(cat_cols):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Compare mean/SD and median/IQR of important variables by fraud label
important = ["Claim_Amount", "Approved_Amount", "Number_of_Claims_Per_Provider_Monthly",
             "Days_Between_Service_and_Claim", "Prior_Visits_12m"]

fig, axes = plt.subplots(1, len(important), figsize=(18, 4))

for ax, col in zip(axes, important):
    g = df.groupby("Is_Fraud")[col]
    mean, sd = g.mean(), g.std()
    median, q1, q3 = g.median(), g.quantile(0.25), g.quantile(0.75)
    x = np.array([0, 1])

    ax.errorbar(x - 0.1, mean, yerr=sd, fmt="o", color="tab:blue", capsize=4, label="Mean ± SD")
    ax.errorbar(x + 0.1, median, yerr=[median - q1, q3 - median], fmt="s", color="tab:orange", capsize=4, label="Median + IQR")

    ax.set_xticks(x, ["No Fraud", "Fraud"])
    ax.set_xlim(-0.5, 1.5)
    ax.set_title(col)

axes[0].legend()
plt.tight_layout()
plt.show()


In [ ]:
# Plot fraud rate for each category
fig, axes = plt.subplots(3, 3, figsize=(15, 12))

for ax, col in zip(axes.flat, cat_cols):
    sns.barplot(data=df, x=col, y="Is_Fraud", errorbar=None, ax=ax)
    ax.axhline(df["Is_Fraud"].mean(), color="red", linestyle="--",
               label=f"Overall fraud rate ({df['Is_Fraud'].mean():.1%})")

    ax.set_title(col)
    ax.set_ylabel("Fraud rate")
    ax.tick_params(axis="x", rotation=45)

for ax in axes.flat[len(cat_cols):]:
    ax.axis("off")
axes.flat[0].legend()
plt.tight_layout()
plt.show()


In [ ]:
# Boxplots to check for outliers by fraud label
fig, axes = plt.subplots(3, 3, figsize=(15, 12))

for ax, col in zip(axes.flat, num_cols):
    sns.boxplot(data=df, x="Is_Fraud", y=col, ax=ax)
    ax.set_xticks([0, 1], ["No Fraud", "Fraud"])
    ax.set_xlabel("")
    ax.set_title(col)

plt.tight_layout()
plt.show()


In [ ]:
# Plot claim vs approved amount by fraud label
plt.figure(figsize=(8,5))
sns.scatterplot(
    data=df, 
    x='Claim_Amount',
    y='Approved_Amount',
    hue='Is_Fraud',
    palette='coolwarm'
)
plt.title("Claim vs Approved Amount")
plt.show()

In [ ]:
# Create approval ratio and plot by fraud label
df["Approval_Ratio"] = df["Approved_Amount"] / df["Claim_Amount"]

plt.figure(figsize=(8, 5))
sns.histplot(data=df, x="Approval_Ratio", hue="Is_Fraud", stat="density", common_norm=False)
plt.title("Approval Ratio by Fraud Label")
plt.show()

In [ ]:
# Flag claims submitted within a week of service
df["Claimed_Within_Week"] = (df["Days_Between_Service_and_Claim"] <= 7).astype(int)

plt.figure(figsize=(6, 4))
sns.barplot(data=df, x="Claimed_Within_Week", y="Is_Fraud", errorbar=None)
plt.xticks([0, 1], ["More than a week", "Within a week"])
plt.xlabel("")
plt.ylabel("Fraud rate")
plt.title("Fraud Rate by Claim Within a Week of Service")
plt.show()

In [ ]:
# Flag claims above the claim amount that gives a 5% FPR
threshold = df.loc[df["Is_Fraud"] == 0, "Claim_Amount"].quantile(0.95)
df["High_Claim_Amount"] = (df["Claim_Amount"] > threshold).astype(int)

plt.figure(figsize=(6, 4))
sns.barplot(data=df, x="High_Claim_Amount", y="Is_Fraud", errorbar=None)
plt.xticks([0, 1], [f"≤ ${threshold:,.0f}", f"> ${threshold:,.0f}"])
plt.xlabel("Claim Amount")
plt.ylabel("Fraud rate")
plt.title("Fraud Rate by High Claim Amount (5% FPR)")
plt.show()